In [ ]:
# Import the libraries we need for this worked example

import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree


In [ ]:
# Find all CSV files in the 'ground_truth' folder
file_list = sorted(glob.glob("data/ground_truth/*.csv"))

# Print each file name so we can check what has been found
for file_name in file_list:
    print(file_name)


In [ ]:
# Create a list to hold the DataFrame from each file
dataframes = []

# Read each CSV file and add its DataFrame to the list
for file_name in file_list:
    temp_df = pd.read_csv(file_name)
    dataframes.append(temp_df)

# Combine all DataFrames into one larger DataFrame
df_blinks = pd.concat(dataframes, ignore_index=True)


In [ ]:
# Show the size of the combined dataset
print("Combined dataset shape:\n", df_blinks.shape)

# Show the column names
print("Column names:\n", df_blinks.columns.tolist())

# Show the first 10 rows
print("First 10 rows:\n", df_blinks.head(10))


In [ ]:
# Count how many missing values there are in each column
# isna() marks missing values as True
# sum() then counts how many True values appear in each column
missing_values = df_blinks.isna().sum()

print(missing_values)


In [ ]:
# Remove rows with missing values and make a copy of the cleaned result
df_blinks_c = df_blinks.dropna().copy()

# Reset the index so the cleaned DataFrame has a simple continuous row number
df_blinks_c = df_blinks_c.reset_index(drop=True)

# Check the shape after cleaning
print("Post-cleaning shape:", df_blinks_c.shape)


In [ ]:
# Create new columns for each stage of the blink event
df_blinks_c["closing_duration"] = df_blinks_c["Eye_closed"] - df_blinks_c["Eye_starts_closing"]
df_blinks_c["closed_duration"] = df_blinks_c["Eye_starts_opening"] - df_blinks_c["Eye_closed"]
df_blinks_c["opening_duration"] = df_blinks_c["Eye_open"] - df_blinks_c["Eye_starts_opening"]

# Show summary statistics for each new duration column
print("Summary of closing duration:\n", df_blinks_c["closing_duration"].describe())
print("\nSummary of closed duration:\n", df_blinks_c["closed_duration"].describe())
print("\nSummary of opening duration:\n", df_blinks_c["opening_duration"].describe())


In [ ]:
# Choose the features we want to visualise
distribution_features = ["closing_duration", "closed_duration", "opening_duration"]

# Find one shared x-axis range across all three features
x_min = df_blinks_c[distribution_features].min().min()
x_max = df_blinks_c[distribution_features].max().max()

# Create one shared set of bins so the histograms are easier to compare
shared_bins = np.linspace(x_min, x_max, 51)

plt.figure(figsize=(10, 12))

# Loop through the features and make one histogram for each
for i, feature in enumerate(distribution_features, start=1):
    plt.subplot(len(distribution_features), 1, i)
    df_blinks_c[feature].hist(bins=shared_bins)

    plt.xlim(x_min, x_max)
    plt.xlabel(feature)
    plt.ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
# Separate the input features from the label column
X = df_blinks_c[["closing_duration", "closed_duration", "opening_duration"]].copy()
y = df_blinks_c["Blink_type"]


In [ ]:
# Count how many examples we have of each blink type
print(df_blinks_c["Blink_type"].value_counts())

# Plot the class counts as a bar chart
df_blinks_c["Blink_type"].value_counts().plot(kind="bar")
plt.ylabel("Count")
plt.show()


In [ ]:
# Split the features and labels into training and test sets so we can train the model on one part of the data and evaluate it on unseen data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
# Create a simple decision tree classifier
# max_depth keeps the tree small so it is easier to understand when plotted
tree_model = DecisionTreeClassifier(max_depth=3, random_state=0)

# Train the model using the training data
tree_model.fit(X_train, y_train)


In [ ]:
# Use the trained tree to predict the labels of the test data
y_pred_tree = tree_model.predict(X_test)

# Calculate the overall accuracy
tree_accuracy = accuracy_score(y_test, y_pred_tree)

print("Decision Tree accuracy:", tree_accuracy)

# Display a confusion matrix for the decision tree predictions
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tree)
plt.title("Decision Tree Confusion Matrix")
plt.show()

# Plot the trained decision tree
plt.figure(figsize=(12, 8))
plot_tree(tree_model, feature_names=X.columns, class_names=tree_model.classes_, filled=True)
plt.show()


In [ ]:
# Create a random forest classifier
forest_model = RandomForestClassifier(n_estimators= 25, max_depth= 3, random_state= 0)

# Set up leave-one-out cross-validation
cv = LeaveOneOut()

# Get one cross-validated prediction for every sample
y_pred_forest = cross_val_predict(forest_model, X, y, cv=cv)

# Calculate the overall leave-one-out accuracy from those predictions
forest_accuracy = accuracy_score(y, y_pred_forest)
print("Mean leave-one-out accuracy:", forest_accuracy)

# Display a confusion matrix using those cross-validated predictions
ConfusionMatrixDisplay.from_predictions(y, y_pred_forest)
plt.show()
